<a href="https://colab.research.google.com/github/LeonimerMelo/Reinforcement-Learning/blob/Policy-Gradient/TRPO_(Trust_Region_Policy_Optimization)_Policy_Gradient_method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução ao TRPO (Trust Region Policy Optimization) - Policy Gradient method

O **TRPO (Trust Region Policy Optimization)** é um algoritmo de **Policy Gradient** proposto por pesquisadores da University of California, Berkeley em 2015. Seu objetivo é resolver um problema comum dos métodos de gradiente de política: **atualizações muito grandes da política podem piorar drasticamente o desempenho do agente**.

A ideia central do TRPO é simples:

> "Melhore a política, mas sem mudar demais a cada atualização."

Essa restrição torna o treinamento mais estável e previsível.

---

## 1. Relembrando Policy Gradient

Nos métodos de Policy Gradient, temos uma política parametrizada:

$$\pi_{\theta}(a|s)$$

onde:

* ($s$) = estado
* ($a$) = ação
* ($\theta$) = parâmetros da rede neural

Nosso objetivo é maximizar o retorno esperado:

$$
J(\theta)=
\mathbb{E}
\left[
\sum_{t=0}^{\infty}
\gamma^t r_t
\right]
$$

O gradiente da política é:

$$
\nabla_\theta J(\theta)
=
\mathbb{E}
\left[
\nabla_\theta \log \pi_\theta(a|s)
A(s,a)
\right]
$$

onde ($A(s,a)$) é a vantagem (*advantage*).

---

##  2. O Problema dos Métodos Tradicionais

Imagine que um agente aprendeu uma política razoavelmente boa.

Após calcular o gradiente, ele atualiza:

$$
\theta_{novo}
=
\theta_{antigo}
+
\alpha \nabla_\theta J(\theta)
$$

Se o passo ($\alpha$) for grande demais:

* a política muda muito;
* ações anteriormente boas deixam de ser escolhidas;
* o desempenho pode cair drasticamente.

Visualmente:

```
Política Boa
      |
      v
Atualização muito grande
      |
      v
Política ruim
```

Esse fenômeno era comum em algoritmos como:

* REINFORCE
* Vanilla Policy Gradient
* Actor-Critic clássico

---

##  3. A Ideia do TRPO

TRPO introduz uma região de confiança (*Trust Region*).

Em vez de perguntar:

> "Qual atualização melhora mais a recompensa?"

ele pergunta:

> "Qual atualização melhora a recompensa sem alterar muito a política atual?"

---

##  4. O Que Significa "Não Alterar Muito"?

Precisamos medir a distância entre duas políticas: $\pi_{old}$ e $\pi_{new}$

Para isso o TRPO utiliza a divergência de Kullback-Leibler (KL).

---

##  Divergência KL

$$D_{KL}(\pi_{old}|\pi_{new})$$

Intuitivamente:

* KL pequena → políticas parecidas
* KL grande → políticas muito diferentes

Exemplo:

| Situação         | KL    |
| ---------------- | ----- |
| Mudança pequena  | 0.001 |
| Mudança moderada | 0.01  |
| Mudança grande   | 0.5   |

TRPO impõe:

$$
D_{KL}
(\pi_{old},\pi_{new})
\leq \delta
$$

onde (\delta) é um valor pequeno.

---

##  5. Função Objetivo do TRPO

Em Policy Gradient tradicional:

$$
\max J(\theta)
$$

No TRPO:

$$
\max_{\theta}
L(\theta)
$$

sujeito a:

$$
D_{KL}
(\pi_{old},\pi_\theta)
\leq \delta
$$

Ou seja:

**Maximizar a melhoria da política respeitando um limite de mudança.**

---

##  6. A Função Surrogate

TRPO utiliza uma aproximação chamada função surrogate:

$$
L(\theta)
=
\mathbb{E}
\left[
\frac{\pi_\theta(a|s)}
{\pi_{old}(a|s)}
A(s,a)
\right]
$$

Observe o termo:

$$
\frac{\pi_\theta(a|s)}
{\pi_{old}(a|s)}
$$

chamado de importance sampling ratio.

Ele mede quanto a nova política mudou em relação à antiga.

---

##  Interpretação do Ratio

Se:

$$
r(\theta)
=
\frac{\pi_\theta(a|s)}
{\pi_{old}(a|s)}
$$

então:

* (r=1) → nada mudou
* (r>1) → ação ficou mais provável
* (r<1) → ação ficou menos provável

---

##  7. O Problema de Otimização

O TRPO resolve:

$$
\max_\theta
\mathbb{E}
\left[
r(\theta)A
\right]
$$

sujeito a

$$
\mathbb{E}
\left[
D_{KL}
(\pi_{old},\pi_\theta)
\right]
\leq \delta
$$

Essa é uma otimização com restrição.

---

##  8. Como o TRPO Resolve Isso?

Resolver diretamente seria muito caro.

TRPO utiliza duas aproximações:

### Aproximação de primeira ordem

Para a função objetivo.

### Aproximação de segunda ordem

Para a restrição KL.

O resultado é um problema quadrático:

$$
\max g^T x
$$

sujeito a

$$
x^T H x
\leq \delta
$$

onde:

* (g) = gradiente
* (H) = Hessiana da KL

---

##  9. Conjugate Gradient

Computar a Hessiana inteira é inviável para redes neurais grandes.

TRPO usa:

### Conjugate Gradient (CG)

para calcular aproximadamente:

$$
H^{-1}g
$$

sem armazenar toda a Hessiana.

Isso reduz drasticamente o custo computacional.

---

##  10. Line Search

Mesmo após encontrar uma direção ótima, o algoritmo verifica:

1. A KL continua pequena?
2. O desempenho melhorou?

Se não:

* reduz o passo;
* tenta novamente.

Esse procedimento é chamado de **backtracking line search**.

---

### Fluxo Completo do TRPO

```text
Coletar trajetórias

        ↓

Calcular vantagens

        ↓

Calcular gradiente

        ↓

Construir restrição KL

        ↓

Conjugate Gradient

        ↓

Determinar direção ótima

        ↓

Line Search

        ↓

Atualizar política
```

---

##  11. Vantagens do TRPO

### Estabilidade

Muito mais estável que REINFORCE.

### Atualizações seguras

Evita destruir uma política já boa.

### Excelente desempenho

Foi um dos primeiros algoritmos capazes de controlar sistemas contínuos complexos.

Exemplos clássicos:

* HalfCheetah
* Hopper
* Walker2D
* Humanoid

---

##  12. Desvantagens

### Complexidade matemática

Envolve:

* Hessiana
* KL divergence
* Constrained Optimization
* Conjugate Gradient

### Implementação difícil

Muito mais complicada que PPO.

### Alto custo computacional

Necessita resolver um problema de otimização a cada atualização.

---

##  13. Relação entre TRPO e PPO

O algoritmo PPO foi criado justamente para simplificar o TRPO.

Comparação:

| Característica         | TRPO       | PPO        |
| ---------------------- | ---------- | ---------- |
| Restrição explícita KL | Sim        | Não        |
| Conjugate Gradient     | Sim        | Não        |
| Hessiana               | Sim        | Não        |
| Implementação          | Difícil    | Fácil      |
| Estabilidade           | Muito alta | Alta       |
| Popularidade atual     | Média      | Muito alta |

Por isso, hoje o PPO é frequentemente descrito como:

> "Uma versão simplificada e prática do TRPO."

---

## 14. Intuição Final

Imagine que você está caminhando em uma montanha tentando subir até o ponto mais alto.

Um método de gradiente tradicional faz:

> "Dê um passo na direção da subida."

O TRPO faz:

> "Dê um passo na direção da subida, mas não saia de uma região segura."

Essa "região segura" é a **Trust Region**, definida pela divergência KL.

Graças a essa restrição, o TRPO evita atualizações agressivas e produz um aprendizado muito mais estável, sendo um dos marcos históricos que levaram ao desenvolvimento dos algoritmos modernos de Policy Optimization, especialmente o PPO.


##Referências
- SCHULMAN, John; LEVINE, Sergey; MORITZ, Philipp; JORDAN, Michael I.; ABBEEL, Pieter. Trust Region Policy Optimization. In: INTERNATIONAL CONFERENCE ON MACHINE LEARNING (ICML), 32., 2015, Lille, France. Proceedings of the 32nd International Conference on Machine Learning. Lille: PMLR, 2015. p. 1889–1897.

- SCHULMAN, John; LEVINE, Sergey; MORITZ, Philipp; JORDAN, Michael I.; ABBEEL, Pieter. Trust Region Policy Optimization. arXiv, Ithaca, n. arXiv:1502.05477, 2015. Disponível em: [arXiv - Trust Region Policy Optimization](https://arxiv.org/abs/1502.05477?utm_source=chatgpt.com). Acesso em: 31 maio 2026.

- SUTTON, Richard S.; BARTO, Andrew G. Reinforcement Learning: An Introduction. 2. ed. Cambridge: MIT Press, 2018.

- SCHULMAN, John; WOLSKI, Filip; DHARIWAL, Prafulla; RADFORD, Alec; KLIMOV, Oleg. Proximal Policy Optimization Algorithms. arXiv, Ithaca, n. arXiv:1707.06347, 2017. Disponível em: arXiv - PPO Algorithms. Acesso em: 31 maio 2026.

- KAKADE, Sham M. A Natural Policy Gradient. In: ADVANCES IN NEURAL INFORMATION PROCESSING SYSTEMS (NeurIPS), 14., 2001. Vancouver. Proceedings of NIPS 2001. Cambridge: MIT Press, 2002. p. 1531–1538.

- SZEPESVÁRI, Csaba. Algorithms for Reinforcement Learning. San Rafael: Morgan & Claypool Publishers, 2010.

- NOCEDAL, Jorge; WRIGHT, Stephen J. Numerical Optimization. 2. ed. New York: Springer, 2006.